# 01 — Download raw conflict data

Fetches the frozen sources the panel is built from and records provenance in `data/raw/MANIFEST.json` (URL, version, SHA-256, download time). The manifest is committed; the data files are gitignored.

| File | Source | Role |
|---|---|---|
| `ged261-csv.zip` | UCDP GED 26.1 (events 1989–2025) | Target and conflict-history features |
| `ged261.pdf` | GED 26.1 codebook | Reference |
| `gw_iisystem.dat` | Gleditsch–Ward independent states | Panel universe (entry/exit dates) |
| `gw_microstatessystem.dat` | Gleditsch–Ward microstates | Checked against GED codes (none appear; excluded) |
| `DirectContiguity320.zip` | COW Direct Contiguity 3.2 (1816–2016) | Neighbours, for the neighbour-conflict feature |
| `V-Dem-CY-Core-v16.csv` | V-Dem v16 Country-Year Core — **manual download** | Annual regime covariates |

**Not downloaded:** UCDP Candidate monthly data — preliminary and revised at each annual release, so it is used only for the live November 2026 check, never for modelling.

**V-Dem must be downloaded by hand** (the site uses a download form): [Country-Year Core v16](https://www.v-dem.net/data/the-v-dem-dataset/country-year-v-dem-core-v16/) → CSV → save into `data/raw/` without renaming. The notebook then checks it is present and records its hash.

Re-running is safe: files whose hash matches the manifest are skipped.

## Setup
Makes `src/` importable no matter where the notebook kernel starts.

In [6]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root == root.parent:
        raise FileNotFoundError("Open this notebook from inside the DACSS_ML_FP repo.")
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.paths import ROOT, RAW, MANIFEST
from src.data.download import (SOURCES, MANUAL_SOURCES, download, register_manual,
                               load_manifest, save_manifest)

RAW.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)

repo root: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP


## Download

In [7]:
manifest = load_manifest(MANIFEST)
for name, meta in SOURCES.items():
    manifest[name] = download(name, meta, RAW, manifest)
for name, meta in MANUAL_SOURCES.items():
    manifest[name] = register_manual(name, meta, RAW, manifest)
save_manifest(manifest, MANIFEST)
print(f"\nManifest written: {MANIFEST.relative_to(ROOT)}")

[skip] ged261-csv.zip — already present, hash matches manifest
[skip] ged261.pdf — already present, hash matches manifest
[skip] gw_iisystem.dat — already present, hash matches manifest
[skip] gw_microstatessystem.dat — already present, hash matches manifest
[skip] DirectContiguity320.zip — already present, hash matches manifest
[skip] V-Dem-CY-Core-v16.csv — manual file, hash matches manifest

Manifest written: data\raw\MANIFEST.json


## Check what arrived
The GED zip's contents determine how the panel step reads it. The Gleditsch–Ward files are plain text; the first lines show their delimiter and date format.

In [8]:
import zipfile

with zipfile.ZipFile(RAW / "ged261-csv.zip") as z:
    for info in z.infolist():
        print(f"{info.filename:40s} {info.file_size / 1e6:8.1f} MB")

GEDEvent_v26_1.csv                          274.0 MB


In [9]:
for name in ["gw_iisystem.dat", "gw_microstatessystem.dat"]:
    print(f"--- {name} ---")
    with open(RAW / name, encoding="latin-1") as f:
        for _ in range(4):
            print(repr(f.readline()))

--- gw_iisystem.dat ---
'2\tUSA\tUnited States of America\t01:01:1816\t31:12:2017\n'
'20\tCAN\tCanada\t01:07:1867\t31:12:2017\n'
'31\tBHM\tBahamas\t10:07:1973\t31:12:2017\n'
'40\tCUB\tCuba\t20:05:1902\t31:12:2017\n'
--- gw_microstatessystem.dat ---
'54\tDMA\tDominica\t03:11:1978\t31:12:2017\n'
'55\tGRN\tGrenada\t07:02:1974\t31:12:2017\n'
'56\tSLU\tSaint Lucia\t22:02:1979\t31:12:2017\n'
'57\tSVG\tSaint Vincent and the Grenadines\t27:10:1979\t31:12:2017\n'


In [10]:
with zipfile.ZipFile(RAW / "DirectContiguity320.zip") as z:
    for info in z.infolist():
        print(f"{info.filename:40s} {info.file_size / 1e6:8.2f} MB")

DirectContiguity320/                         0.00 MB
DirectContiguity320/contdir.csv              0.09 MB
DirectContiguity320/contdir.dta              0.16 MB
DirectContiguity320/contdird.csv             3.10 MB
DirectContiguity320/contdird.dta             1.73 MB
DirectContiguity320/contdirs.csv             0.40 MB
DirectContiguity320/contdirs.dta             0.22 MB
DirectContiguity320/Direct Contiguity Codebook.pdf     0.16 MB
